# Spall health index (HI)

Runs the same pipeline as `python spall_hi.py --data ../data --out out`, one step per cell.
The functions are imported from `spall_hi.py`, so the notebook and the script stay in sync.

## About this model

### Where the model is created
There is no single trained network. The pipeline has three fitted pieces, and no damage labels are used to fit any of them. The photo labels (`EYE`) are only used to score the candidates.

| Notebook section | What gets fitted | Fitted on |
|---|---|---|
| **Load 1-min steady-state medians and condition-correct per gear**: `condition_correct(load_minutes(...))` | Per-gear normalisation: a linear regression of each log-CI on temperature and torque, then a robust median/MAD z-score | **Each gear's own** first hours (0.5–6 h). This includes the validation and test gears, since it uses no labels. |
| **Leave-one-gear-out evaluation**: `evaluate(...)` | `HealthyModel` fitted 4 times, once with each training gear held out | The other three training gears' healthy minutes. **For evaluation only**; these fits are discarded. |
| **Fit on all training gears and write submissions**: `hm = HealthyModel().fit(z["train"])` | **The final model** | Pooled healthy minutes of training gears A, B, C and F |

`HealthyModel.fit` (in `spall_hi.py`) contains:
- a Ledoit–Wolf shrinkage covariance of all 12 CIs, which gives `maha`;
- a Ledoit–Wolf covariance of the 4 tooth CIs, which gives `tooth_maha`;
- a PCA keeping 95% of the variance, which gives `spe`.

`hm.score(...)` and `run_end_scores(...)` then apply it to the validation and test gears.

### Input → output
**Input:** `data/<split>/lf_1s.parquet`, one row per second per gear.
- **IDs and time:** `experiment` (one gear), `run_num` (one session between rig stops and inspections), `test_hours`.
- **12 vibration CIs:** `FEATS` in `spall_hi.py`.
- **Operating conditions:** `ctx_temperature`, `ctx_pau_torque`, plus `ctx_pau_speed`, which is only used to filter.
- **Steps:**
  1. Keep only steady operation: speed 1400 ± 20 rpm and torque within 5% of the gear's nominal.
  2. Take 1-minute medians.
  3. Take the log of each CI and correct it for conditions.
  4. Convert to z-scores against the gear's own healthy window.

**Outputs:**
1. **A per-minute HI** with six candidate scores (`maha`, `spe`, `zpos`, `zmax`, `tooth`, `tooth_maha`). Each is roughly "how many healthy standard deviations away from this gear's normal". About 0 means healthy.
2. **A run-end HI**: one number per run (`TAIL_Q` = 90th percentile of the last `TAIL_MIN` = 60 steady minutes), forced never to decrease over a gear's runs (`iso`), because spalls don't heal.
3. **Label-free selection table** (`label_free_eval.csv`): monotonicity, trendability, prognosability and run-1 false-alarm rate per candidate, computed without labels. With `SELECT = "label_free"` the candidate with the best combined fitness is submitted; by default `tooth_maha` is pinned.
4. **Submission files**: `out/validation_submission.csv` and `out/test_submission.csv` with columns `run_id, time, metric`. `metric` is the run-end HI of the selected candidate at the run end nearest each template time.

### How actionable the output is
**What it supports:**
- **Spall detection at run granularity.** On all four training gears the HI jumps clearly in the run where a spall shows in the photos. The jump starts within the run, from minutes to about an hour before it ends. The practical action is to stop at the end of that run and inspect.
- **Trending within one gear.** A sharp rise against that gear's own earlier runs is the most reliable signal.

**What it doesn't support:**
- **No early warning of initiation.** The HI stays flat at every inspection where the photos show initiation only. It detects damage; it doesn't predict it, and there's no remaining-useful-life estimate.
- **No calibrated threshold.** The values aren't comparable across gears. Healthy runs score about 0–1 on Exp-A and Exp-C but about 5–9 on Exp-F. A spall scores about 8 on Exp-B but about 55 on Exp-C. With four training gears, a fixed alarm level like "inspect above 10" would be a guess. It's better to watch for a jump of several times the gear's own recent level.
- **False alarms after restarts.** HI levels shift after rig restarts (Exp-B's sawtooth, Exp-F's drift), which can look like slow damage.

### Acronyms

**General terms**

| Term | Meaning |
|---|---|
| **HI** | Health Index: one number summarising how far a gear is from healthy |
| **CI** | Condition Indicator: a vibration statistic computed from the accelerometer over each 1-s window (`ci_*` columns) |
| **PHM** | Prognostics and Health Management (the data challenge) |
| **Spall** | A pit where material has flaked off a gear tooth surface. **Initiation** is the earlier stage before a spall forms. |
| **Exp-X / Run** | One gear's test / one session between stops. Each run ends with a photo inspection. |
| **LOGO** | Leave-One-Gear-Out cross-validation |

**Signal-processing terms**

| Term | Meaning |
|---|---|
| **TSA** | Time Synchronous Average: vibration averaged over shaft revolutions so the gear-mesh pattern stands out |
| **Difference / residual signal** | The TSA with the regular mesh tones (and, for the residual, the shaft tones) removed, so what's left is mostly local tooth damage |

**The CIs used**

| CI | Meaning |
|---|---|
| **FM0** | Peak-to-peak of the TSA ÷ the sum of the mesh-harmonic amplitudes (coarse damage) |
| **FM4** | Kurtosis of the difference signal (impulsiveness from one damaged tooth) |
| **M6A / M8A** | 6th / 8th normalised moments of the difference signal: more sensitive versions of FM4 |
| **NA4** | Kurtosis of the residual signal, normalised by the running average variance, so it keeps trending as damage spreads |
| **e_ratio** | Energy ratio: RMS of the difference signal ÷ RMS of the regular mesh components |
| **kurtosis/rms/crest _tsa** | Those statistics computed on the TSA. Crest factor = peak ÷ RMS. |
| **ALR, e_op** | Not defined in `data/channels.csv`. ALR has `alr_bandwidth` and `max_alr_harmonic` settings, so it's some harmonic/sideband ratio. `e_op` looks like a derived signal that kurtosis/RMS/crest/skew are computed on. Check the challenge documentation before describing either in a report. |
| **\*star** (e.g. FM4star) | The same CI baselined on a reference run from another rig experiment. Excluded here. |

**Model-scoring terms**

| Term | Meaning |
|---|---|
| **maha** | Mahalanobis distance: distance from healthy that accounts for correlations between CIs |
| **SPE** | Squared Prediction Error (Q-statistic): what the PCA of healthy data can't reconstruct |
| **PCA** | Principal Component Analysis |
| **Ledoit–Wolf** | A shrinkage covariance estimate, stable with few samples |
| **MAD** | Median Absolute Deviation, a robust standard deviation |
| **zpos / zmax** | RMS of the positive z-scores / largest absolute z-score across CIs |
| **tooth** | Mean positive z of FM4/M6A/M8A/NA4 |
| **tooth_maha** | Mahalanobis distance on those four CIs, set to 0 when they are below healthy on average |
| **iso / cummax** | Isotonic regression (best non-decreasing fit) / running maximum, the two ways of keeping the HI monotone |
| **MSE / Spearman** | Mean squared error after the best monotone map / rank correlation with photo damage |

**Sensors and data**

| Term | Meaning |
|---|---|
| **PAU** | The rig's speed/torque channel (`ctx_pau_*`). The data doesn't expand the name. |
| **DM4500** | Oil-line particle counter. `dm4500_bin_2..7` count the larger ferrous particles, used for the `debris` baseline. |
| **EYE** | The variable holding the by-eye photo damage labels (initiation = 1, spall = 2 per tooth) |

In [ ]:
import json
from pathlib import Path

import pandas as pd

from spall_hi import (
    BASE_H,
    BREAK_IN_H,
    FEATS,
    HI_DEFAULT,
    SPLITS,
    TAIL_MIN,
    TAIL_Q,
    TOOTH,
    HealthyModel,
    condition_correct,
    evaluate,
    label_free_table,
    load_debris,
    load_minutes,
    logo_scores,
    run_end_scores,
    run_ends,
)

pd.set_option("display.width", 200)

## Arguments
Equivalent to `--data ../data --out out` with the script's defaults for `--monotone`, `--hi` and `--select`.

`SELECT` picks the submitted HI: `"fixed"` submits `HI` (pinned to `tooth_maha`), `"label_free"` takes the best label-free fitness, and `"photo"` takes the lowest error against the photo labels (that one uses labels for model selection).

In [ ]:
DATA = Path("../data")
OUT = Path("out")
MONOTONE = "iso"  # one of "none", "cummax", "iso"
HI = HI_DEFAULT  # HI column to submit with SELECT = "fixed"
SELECT = "fixed"  # one of "fixed", "label_free", "photo"

OUT.mkdir(parents=True, exist_ok=True)
cols = ["maha", "spe", "zpos", "zmax", "tooth", "tooth_maha"]

## Load 1-min steady-state medians and condition-correct per gear

In [ ]:
z = {s: condition_correct(load_minutes(DATA, s)) for s in SPLITS}
ends = {s: run_ends(DATA, s) for s in SPLITS}
debris = load_debris(DATA, "train")

## Leave-one-gear-out evaluation vs photo labels
Lower MSE / higher Spearman is better.

In [ ]:
for mode in ["none", "cummax", "iso"]:
    t, _ = evaluate(z["train"], ends["train"], debris, cols, mode)
    print(f"\n[{mode}] leave-one-gear-out vs photo labels (lower mse / higher spearman is better):")
    print(t.to_string())

In [ ]:
table, r_tr = evaluate(z["train"], ends["train"], debris, cols, MONOTONE)
print(r_tr[["experiment", "run_num", "end_h", "photo"] + cols].round(2).to_string(index=False))
s_tr, thr = logo_scores(z["train"])
lf = label_free_table(s_tr, thr, ends["train"], cols)
print("\nlabel-free selection criteria (no labels; higher fitness is better):")
print(lf.to_string())
best = {
    "fixed": HI,
    "label_free": lf.fitness.idxmax(),
    "photo": table.loc[cols, "mse_pooled"].idxmin(),
}[SELECT]
print(f"\nsubmitting HI = {best}  (--select {SELECT})")

## Fit on all training gears and write submissions

In [ ]:
hm = HealthyModel().fit(z["train"])
for split, tmpl in [("validation", "example_validation_submission.csv"), ("test", "example_submission.csv")]:
    r = run_end_scores(hm.score(z[split]), ends[split], cols, MONOTONE)
    sub = pd.read_csv(DATA / split / tmpl)
    r["run_id"] = r.experiment.str[-1]
    # match each template time to the nearest run end of that gear
    sub["metric"] = [
        r.loc[(r.run_id == g), [best, "end_h"]].pipe(lambda x: x[best].iloc[(x.end_h - t).abs().argmin()])
        for g, t in zip(sub.run_id, sub.time)
    ]
    sub["metric"] = sub.metric.round(4)
    sub.to_csv(OUT / f"{split}_submission.csv", index=False)
    print(f"\n{split}:\n{sub.to_string(index=False)}")

In [ ]:
table.to_csv(OUT / "logo_eval.csv")
lf.to_csv(OUT / "label_free_eval.csv")
json.dump(
    {
        "hi": best,
        "select": SELECT,
        "monotone": MONOTONE,
        "feats": FEATS,
        "tooth": TOOTH,
        "base_h": [BREAK_IN_H, BASE_H],
        "tail_min": TAIL_MIN,
        "tail_q": TAIL_Q,
    },
    open(OUT / "config.json", "w"),
    indent=1,
)
table